In [1]:
%pip install Flask Authlib python-dotenv requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import flask
import authlib
import dotenv
import requests

print("Flask:", flask.__version__)
print("Authlib:", authlib.__version__)
print("SecureNova environment ready!")

Flask: 3.1.3
Authlib: 1.9.5
SecureNova environment ready!


C:\Users\ganta.murali.krishna\AppData\Local\Temp\ipykernel_31372\853780168.py:6: DeprecationWarning: The '__version__' attribute is deprecated and will be removed in Flask 3.2. Use feature detection or 'importlib.metadata.version("flask")' instead.
  print("Flask:", flask.__version__)


In [2]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: C:\Users\ganta.murali.krishna\SecureNovaAI\venv\Scripts\python.exe
Python version: 3.14.10 (tags/v3.14.10:b703b2c, Oct 25 2026, 17:36:52) [MSC v.1944 64 bit (AMD64)]


In [18]:

import os
import secrets
from getpass import getpass

# Auth0 application details
AUTH0_DOMAIN = input("Enter your Auth0 Domain: ").strip()
AUTH0_DOMAIN = AUTH0_DOMAIN.replace("https://", "").rstrip("/")

AUTH0_CLIENT_ID = input("Enter your Regular Web App Client ID: ").strip()

AUTH0_CLIENT_SECRET = getpass("Enter your Auth0 Client Secret: ")

AUTH0_API_AUDIENCE = "https://api.securenova.com"

# Generate a secure Flask session key
FLASK_SECRET_KEY = secrets.token_urlsafe(32)

print("\nSecureNova configuration loaded successfully!")
print("Auth0 Domain:", AUTH0_DOMAIN)
print("API Audience:", AUTH0_API_AUDIENCE)
print("Client Secret: Hidden")


Enter your Auth0 Domain:  dev-wuaza68d7rbydra8.us.auth0.com
Enter your Regular Web App Client ID:  2Ae7PwrFePk9PI075AQATAcPda0lvce3
Enter your Auth0 Client Secret:  ········



SecureNova configuration loaded successfully!
Auth0 Domain: dev-wuaza68d7rbydra8.us.auth0.com
API Audience: https://api.securenova.com
Client Secret: Hidden


In [19]:

from flask import Flask, redirect, session, url_for
from authlib.integrations.flask_client import OAuth
from urllib.parse import urlencode
from html import escape

# Initialize Flask
app = Flask(__name__)
app.secret_key = FLASK_SECRET_KEY

app.config.update(
    SESSION_COOKIE_HTTPONLY=True,
    SESSION_COOKIE_SAMESITE="Lax",
    SESSION_COOKIE_SECURE=False
)

# Configure Auth0
oauth = OAuth(app)

oauth.register(
    name="auth0",
    client_id=AUTH0_CLIENT_ID,
    client_secret=AUTH0_CLIENT_SECRET,
    server_metadata_url=(
        f"https://{AUTH0_DOMAIN}/.well-known/openid-configuration"
    ),
    client_kwargs={
        "scope": "openid profile email read:ai-data",
        "code_challenge_method": "S256"
    }
)

# Homepage
@app.route("/")
def home():
    if session.get("user"):
        return """
        <h1>Welcome to SecureNova AI</h1>
        <p>You are logged in successfully.</p>
        <a href="/dashboard">Open Dashboard</a><br><br>
        <a href="/logout">Logout</a>
        """

    return """
    <h1>Welcome to SecureNova AI</h1>
    <p>Secure AI Chat Application</p>
    <a href="/login">Login with Auth0</a>
    """

# Login using Authorization Code + PKCE
@app.route("/login")
def login():
    redirect_uri = "http://localhost:5000/callback"

    return oauth.auth0.authorize_redirect(
        redirect_uri,
        audience=AUTH0_API_AUDIENCE
    )

# Auth0 callback
@app.route("/callback")
def callback():
    token = oauth.auth0.authorize_access_token()

    user_info = token.get("userinfo")

    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    session["user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "email": user_info.get("email", ""),
        "sub": user_info.get("sub", "")
    }

    return redirect(url_for("dashboard"))

# Protected dashboard
@app.route("/dashboard")
def dashboard():
    user = session.get("user")

    if not user:
        return redirect(url_for("login"))

    name = escape(user["name"])
    email = escape(user["email"])
    subject = escape(user["sub"])

    return f"""
    <h1>SecureNova AI Dashboard</h1>
    <p>Welcome, {name}</p>
    <p>Email: {email}</p>
    <p>Auth0 User ID: {subject}</p>
    <a href="/">Home</a><br><br>
    <a href="/logout">Logout</a>
    """

# Logout
@app.route("/logout")
def logout():
    session.clear()

    logout_url = (
        f"https://{AUTH0_DOMAIN}/v2/logout?"
        + urlencode({
            "client_id": AUTH0_CLIENT_ID,
            "returnTo": "http://localhost:5000/"
        })
    )

    return redirect(logout_url)

print("SecureNova Flask application created!")
print("PKCE method: S256")
print("Routes: Home, Login, Callback, Dashboard, Logout")


SecureNova Flask application created!
PKCE method: S256
Routes: Home, Login, Callback, Dashboard, Logout


127.0.0.1 - - [10/Nov/2026 16:51:05] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 16:51:10] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 16:51:13] "GET /callback?code=KfxtCEK6HmEeWyiT0M4tVZH9T36GQT2TCVAnV0c9SJPSO&state=sxE1LQpYtHgeZHr6ovgZ9CuEL7txFG&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 16:51:13] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 16:52:31] "GET /api/user-data HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 16:53:22] "GET /api/admin HTTP/1.1" 403 -


In [5]:

from werkzeug.serving import make_server
from threading import Thread

# Start Flask server
server = make_server("127.0.0.1", 5000, app)

server_thread = Thread(
    target=server.serve_forever,
    daemon=True
)

server_thread.start()

print("SecureNova AI server started successfully!")
print("Application URL: http://localhost:5000")


SecureNova AI server started successfully!
Application URL: http://localhost:5000


In [8]:
%pip install "PyJWT[crypto]"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import jwt
from flask import jsonify
from jwt import PyJWKClient

# Temporary server-side token storage for this demonstration
token_store = {}

# Auth0 public keys used to verify JWT signatures
jwks_client = PyJWKClient(
    f"https://{AUTH0_DOMAIN}/.well-known/jwks.json"
)

print("JWT verification configuration ready!")

JWT verification configuration ready!


In [22]:
def callback_updated():
    token = oauth.auth0.authorize_access_token()

    user_info = token.get("userinfo")

    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    # Generate a random identifier for the server-side token
    token_session_id = secrets.token_urlsafe(32)

    # Keep the access token on the server
    token_store[token_session_id] = token["access_token"]

    # Store only the random identifier in the Flask session
    session["token_session_id"] = token_session_id

    session["user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "email": user_info.get("email", ""),
        "sub": user_info.get("sub", "")
    }

    return redirect(url_for("dashboard"))


# Replace the existing callback function
app.view_functions["callback"] = callback_updated

print("Callback updated successfully!")

Callback updated successfully!


In [23]:
def verify_current_user_token():

    token_session_id = session.get("token_session_id")

    if not token_session_id:
        return None

    access_token = token_store.get(token_session_id)

    if not access_token:
        return None

    try:
        signing_key = jwks_client.get_signing_key_from_jwt(
            access_token
        )

        claims = jwt.decode(
            access_token,
            signing_key.key,
            algorithms=["RS256"],
            audience=AUTH0_API_AUDIENCE,
            issuer=f"https://{AUTH0_DOMAIN}/"
        )

        return claims

    except Exception as error:
        print("JWT verification failed:", type(error).__name__)
        return None


print("JWT verification function ready!")

JWT verification function ready!


In [24]:
def api_user_data():

    claims = verify_current_user_token()

    if claims is None:
        return jsonify({
            "error": "Invalid or expired access token"
        }), 401

    scopes = claims.get("scope", "").split()

    if "read:ai-data" not in scopes:
        return jsonify({
            "error": "Insufficient permissions"
        }), 403

    return jsonify({
        "message": "User data accessed successfully!",
        "scope": scopes,
        "access": "READ"
    }), 200


def api_admin():

    claims = verify_current_user_token()

    if claims is None:
        return jsonify({
            "error": "Invalid or expired access token"
        }), 401

    scopes = claims.get("scope", "").split()

    if "write:admin" not in scopes:
        return jsonify({
            "error": "Forbidden: Admin permission required"
        }), 403

    return jsonify({
        "message": "Admin access granted!",
        "access": "ADMIN"
    }), 200


# Register the endpoints
app.add_url_rule(
    "/api/user-data",
    endpoint="api_user_data",
    view_func=api_user_data
)

app.add_url_rule(
    "/api/admin",
    endpoint="api_admin",
    view_func=api_admin
)

print("Protected API endpoints created!")
print("GET /api/user-data")
print("GET /api/admin")

Protected API endpoints created!
GET /api/user-data
GET /api/admin


In [17]:
server.shutdown()
server_thread.join(timeout=5)

print("Flask server stopped successfully!")

Flask server stopped successfully!


In [25]:
# Register the protected API endpoints

app.add_url_rule(
    "/api/user-data",
    endpoint="api_user_data",
    view_func=api_user_data,
    methods=["GET"]
)

app.add_url_rule(
    "/api/admin",
    endpoint="api_admin",
    view_func=api_admin,
    methods=["GET"]
)

print("API routes registered successfully!")

API routes registered successfully!


In [26]:
def logout_updated():

    token_session_id = session.get("token_session_id")

    if token_session_id:
        token_store.pop(token_session_id, None)

    session.clear()

    logout_url = (
        f"https://{AUTH0_DOMAIN}/v2/logout?"
        + urlencode({
            "client_id": AUTH0_CLIENT_ID,
            "returnTo": "http://localhost:5000/"
        })
    )

    return redirect(logout_url)


app.view_functions["logout"] = logout_updated

print("Secure logout configured!")

Secure logout configured!


In [27]:
from werkzeug.serving import make_server
from threading import Thread

server = make_server("127.0.0.1", 5000, app)

server_thread = Thread(
    target=server.serve_forever,
    daemon=True
)

server_thread.start()

print("SecureNova AI server started!")
print("http://localhost:5000")


SecureNova AI server started!
http://localhost:5000


In [14]:
server.shutdown()
server_thread.join(timeout=5)

print("Flask server stopped successfully!")

Flask server stopped successfully!


In [28]:
print("Auth0 Domain:", AUTH0_DOMAIN)
print("Client ID ending:", AUTH0_CLIENT_ID[-6:])
print("Client Secret loaded:", bool(AUTH0_CLIENT_SECRET))
print("Client Secret length:", len(AUTH0_CLIENT_SECRET))

Auth0 Domain: dev-wuaza68d7rbydra8.us.auth0.com
Client ID ending: 0lvce3
Client Secret loaded: True
Client Secret length: 64


In [29]:

from datetime import datetime, timezone

# Get the latest access token stored by the Flask callback
if not token_store:
    print("No access token found. Please log in first.")
else:
    access_token = list(token_store.values())[-1]

    # Verify the JWT using the existing verification function
    claims = verify_current_user_token()

    if claims is None:
        print("JWT verification failed.")
    else:
        header = jwt.get_unverified_header(access_token)

        print("========== JWT VERIFICATION ==========")
        print("Signature Algorithm:", header.get("alg"))
        print("Issuer (iss):", claims.get("iss"))
        print("Audience (aud):", claims.get("aud"))
        print("Expiration (exp):", datetime.fromtimestamp(
            claims["exp"], tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC"))
        print("Scopes:", claims.get("scope"))
        print("======================================")
        print("JWT signature, issuer, audience and expiry verified.")


No access token found. Please log in first.


127.0.0.1 - - [10/Nov/2026 16:58:32] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 16:58:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 16:58:46] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 16:58:54] "GET /callback?code=8hkLKNBK3kraKG6uBxzbn5JSWqyqxZ5YxFx0Ejx0BPyhd&state=ledrMulgL7Pr43KPSwu9BF1G9m6nKT&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 16:58:54] "GET /dashboard HTTP/1.1" 200 -


In [30]:
print("Number of stored tokens:", len(token_store))

if token_store:
    print("Access token successfully stored!")
else:
    print("No token stored. Callback needs checking.")

Number of stored tokens: 1
Access token successfully stored!


In [31]:

from datetime import datetime, timezone
import jwt

# Retrieve the latest stored token
token_session_id = list(token_store.keys())[-1]
access_token = token_store[token_session_id]

try:
    # Get Auth0's public signing key
    signing_key = jwks_client.get_signing_key_from_jwt(access_token)

    # Verify JWT signature and claims
    claims = jwt.decode(
        access_token,
        signing_key.key,
        algorithms=["RS256"],
        audience=AUTH0_API_AUDIENCE,
        issuer=f"https://{AUTH0_DOMAIN}/"
    )

    # Read JWT header
    header = jwt.get_unverified_header(access_token)

    print("========== JWT VERIFICATION ==========")
    print("Signature Algorithm:", header.get("alg"))
    print("Issuer (iss):", claims.get("iss"))
    print("Audience (aud):", claims.get("aud"))
    print(
        "Expiration (exp):",
        datetime.fromtimestamp(
            claims["exp"], tz=timezone.utc
        ).strftime("%Y-%m-%d %H:%M:%S UTC")
    )
    print("Scopes:", claims.get("scope"))
    print("======================================")
    print("JWT signature and claims verified successfully!")

except Exception as error:
    print("JWT verification failed:", type(error).__name__, str(error))


========== JWT VERIFICATION ==========
Signature Algorithm: RS256
Issuer (iss): https://dev-wuaza68d7rbydra8.us.auth0.com/
Audience (aud): ['https://api.securenova.com', 'https://dev-wuaza68d7rbydra8.us.auth0.com/userinfo']
Expiration (exp): 2026-11-11 11:28:54 UTC
Scopes: openid profile email read:ai-data
JWT signature and claims verified successfully!


127.0.0.1 - - [10/Nov/2026 17:06:32] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:06:32] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:06:45] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:06:53] "GET /callback?code=FugiQvaazIQ3sd-JZx85dSbX81gry5xuocI8S4fIHXAhe&state=gClTWCbO3asvz8Lw7MWuj0RWtHjmoN&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:06:53] "GET /dashboard HTTP/1.1" 200 -


In [34]:

id_token_store = {}
latest_token_session_id = None

def callback_with_id_token():
    global latest_token_session_id

    token = oauth.auth0.authorize_access_token()

    user_info = token.get("userinfo")
    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    id_token = token.get("id_token")
    if not id_token:
        raise RuntimeError("Auth0 did not return an ID token.")

    token_session_id = secrets.token_urlsafe(32)

    token_store[token_session_id] = token["access_token"]
    id_token_store[token_session_id] = id_token

    latest_token_session_id = token_session_id

    session["token_session_id"] = token_session_id
    session["user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "email": user_info.get("email", ""),
        "sub": user_info.get("sub", "")
    }

    return redirect(url_for("dashboard"))

app.view_functions["callback"] = callback_with_id_token

print("Updated callback successfully!")


Updated callback successfully!


127.0.0.1 - - [10/Nov/2026 17:11:07] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:11:07] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:11:08] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:11:08] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:11:19] "GET /callback?code=MOV17bPGtujE1bgmrQJ_NO9XPU57ubcNpn9EOuqE1aQt8&state=xb8l5WBphMZe0GMWpcIdq2um78pgpk&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:11:19] "GET /dashboard HTTP/1.1" 200 -


In [35]:

if latest_token_session_id is None:
    print("No token session found. Please log in again.")

elif latest_token_session_id not in id_token_store:
    print("ID token not found.")

else:
    id_token = id_token_store[latest_token_session_id]

    signing_key = jwks_client.get_signing_key_from_jwt(id_token)

    id_claims = jwt.decode(
        id_token,
        signing_key.key,
        algorithms=["RS256"],
        audience=AUTH0_CLIENT_ID,
        issuer=f"https://{AUTH0_DOMAIN}/"
    )

    agent_claim = "https://securenova.com/claims/agent_id"

    print("ID token signature verified.")
    print("Issuer:", id_claims.get("iss"))
    print("Audience:", id_claims.get("aud"))
    print("Agent ID:", id_claims.get(agent_claim))


No token session found. Please log in again.


In [36]:
print("Latest token session ID:", latest_token_session_id)
print("ID tokens stored:", len(id_token_store))
print("Access tokens stored:", len(token_store))

Latest token session ID: None
ID tokens stored: 0
Access tokens stored: 1


In [37]:

id_token_store = {}
latest_token_session_id = None

def callback_with_id_token():
    global latest_token_session_id

    token = oauth.auth0.authorize_access_token()

    user_info = token.get("userinfo")
    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    id_token = token.get("id_token")
    if not id_token:
        raise RuntimeError("Auth0 did not return an ID token.")

    token_session_id = secrets.token_urlsafe(32)

    token_store[token_session_id] = token["access_token"]
    id_token_store[token_session_id] = id_token

    latest_token_session_id = token_session_id

    session["token_session_id"] = token_session_id
    session["user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "email": user_info.get("email", ""),
        "sub": user_info.get("sub", "")
    }

    return redirect(url_for("dashboard"))

app.view_functions["callback"] = callback_with_id_token

print("Callback handler:", app.view_functions["callback"].__name__)
print("Callback updated successfully!")


Callback handler: callback_with_id_token
Callback updated successfully!


127.0.0.1 - - [10/Nov/2026 17:14:49] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:14:49] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:14:54] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:14:56] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:14:57] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:15:03] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:15:11] "GET /callback?code=5iDpb5BBxaQF1KXHX6q8GuvUTNCvst2E9tWgl8IdK4zzo&state=S6S2nlztNNmRl6Ho6X9m2JFBEl4QUy&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:15:11] "GET /dashboard HTTP/1.1" 200 -


In [38]:
print("Latest token session ID exists:",
      latest_token_session_id is not None)

print("ID tokens stored:", len(id_token_store))
print("Access tokens stored:", len(token_store))

Latest token session ID exists: False
ID tokens stored: 0
Access tokens stored: 1


In [39]:
print("Active callback function:",
      app.view_functions["callback"].__name__)

print("Expected callback function: callback_with_id_token")

print("Callback route:")
for rule in app.url_map.iter_rules():
    if rule.endpoint == "callback":
        print(rule)

Active callback function: callback_with_id_token
Expected callback function: callback_with_id_token
Callback route:
/callback


127.0.0.1 - - [10/Nov/2026 17:17:05] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:17:05] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:17:05] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:17:08] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:17:16] "GET /callback?code=1_zXTgUUTPRpstGfBakg9WI1PFIUDRjVaz2fK6Mxh0_3e&state=lBbnW5KI5ToswVBYqtnF6VmMAEArGK&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:17:16] "GET /dashboard HTTP/1.1" 200 -


In [40]:
print("Latest token session ID exists:",
      latest_token_session_id is not None)

print("ID tokens stored:", len(id_token_store))
print("Access tokens stored:", len(token_store))

print("Active callback:",
      app.view_functions["callback"].__name__)

Latest token session ID exists: False
ID tokens stored: 0
Access tokens stored: 1
Active callback: callback_with_id_token


In [41]:

def callback_with_id_token():
    global latest_token_session_id

    print("\n===== CALLBACK DIAGNOSTICS =====")

    token = oauth.auth0.authorize_access_token()

    print("Token exchange successful.")
    print("ID token received:", bool(token.get("id_token")))
    print("Access token received:", bool(token.get("access_token")))
    print("User information received:", bool(token.get("userinfo")))

    user_info = token.get("userinfo")
    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    id_token = token.get("id_token")
    if not id_token:
        raise RuntimeError("Auth0 did not return an ID token.")

    token_session_id = secrets.token_urlsafe(32)

    token_store[token_session_id] = token["access_token"]
    id_token_store[token_session_id] = id_token

    latest_token_session_id = token_session_id

    session["token_session_id"] = token_session_id
    session["user"] = {
        "name": user_info.get("name", "Authenticated User"),
        "email": user_info.get("email", ""),
        "sub": user_info.get("sub", "")
    }

    print("ID token stored:", token_session_id in id_token_store)
    print("Latest session ID assigned:",
          latest_token_session_id is not None)
    print("================================\n")

    return redirect(url_for("dashboard"))


app.view_functions["callback"] = callback_with_id_token

print("Diagnostic callback installed.")


Diagnostic callback installed.


127.0.0.1 - - [10/Nov/2026 17:22:27] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:22:28] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:22:28] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:22:30] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:22:40] "GET /callback?code=HhEn0YdTygso7n-Sg9WtgB2R9h27fEpOYr26m20iDU-i8&state=xTJT8G3cp6UK0RNvUKAvnAAEfYEh6Y&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:22:40] "GET /dashboard HTTP/1.1" 200 -


In [42]:
import inspect

print("Current Flask app:", id(app))
print("Server Flask app:", id(server.app))

print("Active callback:",
      app.view_functions["callback"].__name__)

print("Diagnostic callback:",
      callback_with_id_token.__name__)

print(
    "Same callback:",
    app.view_functions["callback"] is callback_with_id_token
)

print("\nCallback source:")
print(inspect.getsource(app.view_functions["callback"]))

Current Flask app: 2651945549520
Server Flask app: 2651945549520
Active callback: callback_with_id_token
Diagnostic callback: callback_with_id_token
Same callback: True

Callback source:
def callback_with_id_token():
    global latest_token_session_id

    print("\n===== CALLBACK DIAGNOSTICS =====")

    token = oauth.auth0.authorize_access_token()

    print("Token exchange successful.")
    print("ID token received:", bool(token.get("id_token")))
    print("Access token received:", bool(token.get("access_token")))
    print("User information received:", bool(token.get("userinfo")))

    user_info = token.get("userinfo")
    if not user_info:
        raise RuntimeError("Auth0 did not return user information.")

    id_token = token.get("id_token")
    if not id_token:
        raise RuntimeError("Auth0 did not return an ID token.")

    token_session_id = secrets.token_urlsafe(32)

    token_store[token_session_id] = token["access_token"]
    id_token_store[token_session_id] = id_token

In [43]:
callback_debug = {
    "executed": False,
    "id_token_received": False,
    "access_token_received": False,
    "userinfo_received": False,
    "error": None
}

def callback_with_id_token():
    global latest_token_session_id

    callback_debug["executed"] = True

    try:
        token = oauth.auth0.authorize_access_token()

        callback_debug["id_token_received"] = bool(token.get("id_token"))
        callback_debug["access_token_received"] = bool(token.get("access_token"))
        callback_debug["userinfo_received"] = bool(token.get("userinfo"))

        user_info = token.get("userinfo")

        if not user_info:
            raise RuntimeError("User information not received.")

        id_token = token.get("id_token")

        if not id_token:
            raise RuntimeError("ID token not received.")

        token_session_id = secrets.token_urlsafe(32)

        token_store[token_session_id] = token["access_token"]
        id_token_store[token_session_id] = id_token

        latest_token_session_id = token_session_id

        session["token_session_id"] = token_session_id
        session["user"] = {
            "name": user_info.get("name", "Authenticated User"),
            "email": user_info.get("email", ""),
            "sub": user_info.get("sub", "")
        }

        return redirect(url_for("dashboard"))

    except Exception as error:
        callback_debug["error"] = str(error)
        raise

app.view_functions["callback"] = callback_with_id_token

print("Diagnostic callback installed successfully.")

Diagnostic callback installed successfully.


In [44]:
server.shutdown()
server_thread.join(timeout=5)

server = make_server("127.0.0.1", 5000, app)

server_thread = Thread(
    target=server.serve_forever,
    daemon=True
)

server_thread.start()

print("SecureNova AI server restarted successfully!")

SecureNova AI server restarted successfully!


127.0.0.1 - - [10/Nov/2026 17:27:20] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:27:21] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:27:21] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:27:22] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:27:30] "GET /callback?code=rTWfTL37RHjaEIMUBCpp0C7mmNsTbtt9X9EJvmsgcw-dh&state=N4oVTJifKBvQTg64oFQvIs755Y0qMP&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:27:30] "GET /dashboard HTTP/1.1" 200 -


In [53]:
print("Callback executed:", callback_debug["executed"])
print("ID token received:", callback_debug["id_token_received"])
print("Access token received:", callback_debug["access_token_received"])
print("User information received:", callback_debug["userinfo_received"])
print("Error:", callback_debug["error"])

print("ID tokens stored:", len(id_token_store))

Callback executed: False
ID token received: False
Access token received: False
User information received: False
Error: None
ID tokens stored: 0


In [54]:
print("Notebook Flask app ID:", id(app))
print("Running server app ID:", id(server.app))

print("Server thread alive:", server_thread.is_alive())

print("Active callback:",
      app.view_functions["callback"].__name__)

print("Same Flask application:",
      app is server.app)

print("Callback URL registered:",
      any(rule.rule == "/callback" for rule in app.url_map.iter_rules()))

Notebook Flask app ID: 2651945549520
Running server app ID: 2651945549520
Server thread alive: False
Active callback: callback_with_id_token
Same Flask application: True
Callback URL registered: True


In [47]:
print("Registered callback routes:")

for rule in app.url_map.iter_rules():
    if "callback" in rule.rule:
        print("URL:", rule.rule)
        print("Endpoint:", rule.endpoint)
        print("Methods:", rule.methods)
        print("Function:",
              app.view_functions[rule.endpoint].__name__)

print("\nExpected endpoint function:")
print(app.view_functions.get("callback"))

print("\nDiagnostic execution status:")
print(callback_debug)

Registered callback routes:
URL: /callback
Endpoint: callback
Methods: {'HEAD', 'GET', 'OPTIONS'}
Function: callback_with_id_token

Expected endpoint function:
<function callback_with_id_token at 0x0000026974906F00>

Diagnostic execution status:
{'executed': False, 'id_token_received': False, 'access_token_received': False, 'userinfo_received': False, 'error': None}


In [48]:
try:
    server.shutdown()
    server_thread.join(timeout=5)
    print("Old Flask server stopped successfully.")
except Exception as e:
    print("Server already stopped or unavailable:", e)

Old Flask server stopped successfully.


127.0.0.1 - - [10/Nov/2026 17:39:42] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:39:48] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:40:11] "GET /api/user-data HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:40:26] "GET /api/admin HTTP/1.1" 403 -
127.0.0.1 - - [10/Nov/2026 17:43:10] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:43:17] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:43:19] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:43:20] "GET /logout HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:43:22] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [10/Nov/2026 17:43:24] "GET /login HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:43:32] "GET /callback?code=JwxUOq99VIIvB9I_4iw3uDiEtjhDVXkURkmF3nnOfKJQM&state=hoQ8UykZdDMOtN8IGsBZcZr2UQFDTf&iss=https://dev-wuaza68d7rbydra8.us.auth0.com/ HTTP/1.1" 302 -
127.0.0.1 - - [10/Nov/2026 17:43:32] "GET /dashboard HTTP/1.1" 200 -


In [52]:
server.shutdown()
server_thread.join(timeout=5)

print("Old Flask server stopped!")

Old Flask server stopped!
